# Geographic-Scale Training

Sweeps a `sample_idx x k x model x multiplier` grid, training one model per cell and writing `results.json` per run. `sample_idx` is outermost; re-run to add samples. Finished cells are skipped.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os

import pyrootutils

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("TF_XLA_FLAGS", "--tf_xla_auto_jit=0")
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf

tf.get_logger().setLevel('ERROR')

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

In [ ]:
from building.polychirp_training.dataset import load_dataset_info
from building.polychirp_training.runner import ScaleGridConfig, run_scale
from building.models import REGISTRY
print("Available models:", sorted(REGISTRY))

# Parameters

In [ ]:
from building.models import BASE_LIST

COLLECTION = "scale_49_2_r50"
SPECS = BASE_LIST + []  # curated set plus notebook extras

N_SAMPLES = 10
EPOCHS = 50
PATIENCE = 10
BATCH_SIZE = 32
SEED = 42

_info = load_dataset_info(COLLECTION)
K_VALUES = range(1, _info.n_targets + 1)
TASK_SLUG = COLLECTION

print(f"collection: {COLLECTION} ({_info.place.name}, n_targets={_info.n_targets})")
for s in SPECS:
    print(f"  {s.name} x{s.multi:g}")
print(f"k: {list(K_VALUES)}")
print(f"runs under: models/{TASK_SLUG}/<model>/size_x<mult>/k<k>/sample_<i>/")


In [ ]:
config = ScaleGridConfig(
    collection=COLLECTION,
    specs=SPECS,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

# Run the grid

`sample_idx` outermost; re-run to add samples. Finished cells are skipped.

In [ ]:
produced = run_scale(
    config,
    n_samples=N_SAMPLES,
    k_values=K_VALUES,
)
print(f"New runs this pass: {len(produced)}")
for p in produced[-3:]:
    print(p)